In [4]:
# Run in Jupyter for rendered mathematical output.
# Python 3; tested with SymPy 1.14.0: pip install sympy==1.14.0 ipython
#
# A template is (A, D): boundary letters A and integer difference vectors D.
# The empty word is ''.
# S[a] contains (source letter, prefix Parikh vector, suffix Parikh vector).
# B[a][q] contains the splits in S[a] with prefix score q.
# weights and moduli describe the canonical score map; modulus 0 means Z.
# Prepare S, B and the matrices once, then pass them to the searches.
# bounds gives coordinate bounds on each parent difference vector;
# it may also be a function bounds(v). condition(D) adds region inequalities.

from sympy import Matrix, ZZ, Rational, ceiling, floor, latex
from IPython.display import display, Math
from sympy.polys.matrices import DomainMatrix
from sympy.polys.matrices.normalforms import smith_normal_decomp
from itertools import product
from collections import deque
from math import prod, lcm


# ==================== Basic functions ====================

def parikh(word, alphabet):
    return tuple(word.count(a) for a in alphabet)


def frequency_matrix(h, alphabet):
    return Matrix([parikh(h[a], alphabet) for a in alphabet])


def smith_form(M):
    Delta, L, R = smith_normal_decomp(DomainMatrix.from_Matrix(M).convert_to(ZZ))
    return tuple(x.to_Matrix() for x in (Delta, L, R))


def make_template(A, D):
    return tuple(A), tuple(tuple(d) for d in D)


def zero_template(k, alphabet):
    return make_template(['']*(k+1), [(0,)*len(alphabet)]*(k-1))


def print_morphism(h, k):
    images = [rf'{a}\mapsto\mathtt{{{word}}}' for a, word in h.items()]
    display(Math(r'h:\quad ' + r',\quad '.join(images) + rf',\qquad k={k}'))
    print()


# ==================== Splits and score buckets ====================

def split_set(h, a, nontrivial=False):
    splits = [('', '', '', '')] if a == '' else []
    for A, word in h.items():
        if a == '':
            for j in range(len(word)+1):
                if nontrivial and (j == 0 or j == len(word)):
                    continue
                splits.append((A, word[:j], '', word[j:]))
        else:
            for j, letter in enumerate(word):
                if letter == a:
                    splits.append((A, word[:j], a, word[j+1:]))
    return tuple(splits)


def split_sets(h, alphabet, nontrivial=False):
    return {
        a: tuple((A, parikh(p, alphabet), parikh(s, alphabet))
                 for A, p, middle, s in split_set(h, a, nontrivial))
        for a in ('',)+tuple(alphabet)
    }


def normalize_score(v, moduli):
    return tuple(x % q if q else x for x, q in zip(v, moduli))


def canonical_score(v, alphabet, weights, moduli):
    return normalize_score(
        tuple(sum(v[i]*weights[a][j] for i, a in enumerate(alphabet))
              for j in range(len(moduli))),
        moduli
    )


def score_buckets(S, alphabet, weights, moduli):
    prefixes = {p for splits in S.values() for A, p, s in splits}
    prefix_scores = {p: canonical_score(p, alphabet, weights, moduli)
                     for p in prefixes}
    B = {}
    for a, splits in S.items():
        B[a] = {}
        for split in splits:
            B[a].setdefault(prefix_scores[split[1]], []).append(split)
    return B


def nontrivial_buckets(B):
    B1 = {a: dict(buckets) for a, buckets in B.items()}
    B1[''] = {}
    for q, splits in B[''].items():
        remaining = tuple(split for split in splits
                          if split[0] == '' or (any(split[1]) and any(split[2])))
        if remaining:
            B1[''][q] = remaining
    return B1


def admissible_score_sequences(t, B, alphabet, weights, moduli):
    A, D = t
    k = len(A)-1
    d_scores = [canonical_score(d, alphabet, weights, moduli) for d in D]
    for q0 in B[A[0]]:
        for q1 in B[A[1]]:
            scores = [q0, q1]
            for i in range(1, k):
                q = normalize_score(
                    tuple(2*scores[i][j]-scores[i-1][j]
                          - weights[A[i-1]][j]+weights[A[i]][j]+d_scores[i-1][j]
                          for j in range(len(moduli))),
                    moduli
                )
                if q not in B[A[i+1]]:
                    break
                scores.append(q)
            else:
                yield tuple(scores)


def compatible_split_count(t, B, alphabet, weights, moduli):
    return sum(prod(len(B[a][q]) for a, q in zip(t[0], scores))
               for scores in admissible_score_sequences(t, B, alphabet, weights, moduli))


# ==================== Prepare the matrices once ====================

def prepare_inverse(M):
    inverse = M.inv()
    denominator = lcm(*(int(x.q) for x in inverse))
    N = tuple(tuple(int(denominator*inverse[i, j]) for j in range(M.cols))
              for i in range(M.rows))
    return N, denominator  # M^-1 = N / denominator, exactly.


def difference_bounds(h, inverse):
    # For these expanding examples, rho < 1 and this box contains all ancestors.
    N, denominator = inverse
    m = len(N)
    rho = Rational(max(sum(abs(N[i][j]) for i in range(m))
                       for j in range(m)), denominator)
    C = int(ceiling(2*max(map(len, h.values()))*rho/(1-rho)))
    return (C,)*m


def prepare_smith(M):
    Delta, L, R = smith_form(M)
    U = L.inv()
    diagonal = tuple(int(Delta[j, j]) for j in range(M.rows))
    matrices = [tuple(tuple(int(X[i, j]) for j in range(M.cols))
                      for i in range(M.rows)) for X in (L, R, U)]
    return diagonal, *matrices


# ==================== Solve D*M = v ====================

def difference_vectors(v, inverse, bounds, smith=None, condition=None,
                       known_solvable=False):
    m = len(v)
    C = bounds(v) if callable(bounds) else bounds
    if inverse is not None:
        N, denominator = inverse
        numerator = tuple(sum(v[i]*N[i][j] for i in range(m)) for j in range(m))
        if not known_solvable and any(x % denominator for x in numerator):
            return ()
        D = tuple(x // denominator for x in numerator)
        if all(abs(D[j]) <= C[j] for j in range(m)):
            if condition is None or condition(D):
                return (D,)
        return ()

    diagonal, L, R, U = smith
    r = sum(x != 0 for x in diagonal)
    w = tuple(sum(v[i]*R[i][j] for i in range(m)) for j in range(m))
    if not known_solvable:
        if any(w[j] % abs(diagonal[j]) for j in range(r)):
            return ()
        if any(w[j] for j in range(r, m)):
            return ()

    fixed = tuple(w[j] // diagonal[j] for j in range(r))
    free_bounds = [sum(C[i]*abs(U[i][j]) for i in range(m)) for j in range(r, m)]
    solutions = []
    for z in product(*(range(-int(b), int(b)+1) for b in free_bounds)):
        y = fixed+z
        D = tuple(sum(y[i]*L[i][j] for i in range(m)) for j in range(m))
        if all(abs(D[j]) <= C[j] for j in range(m)):
            if condition is None or condition(D):
                solutions.append(D)
    return tuple(solutions)


def parent_rhs(d, left, middle, right):
    return tuple(d[j]+left[2][j]+middle[1][j]-middle[2][j]-right[1][j]
                 for j in range(len(d)))


# ==================== Ordinary parent search ====================

def ordinary_parent_search(t, S, inverse, bounds, smith=None, condition=None):
    A, D = t
    partial = {((), ())}
    for j, a in enumerate(A):
        following = set()
        for chosen, differences in partial:
            for split in S[a]:
                extended = chosen+(split,)
                if j < 2:
                    following.add((extended, differences))
                else:
                    v = parent_rhs(D[j-2], *extended[-3:])
                    for d in difference_vectors(v, inverse, bounds, smith, condition):
                        following.add((extended, differences+(d,)))
        partial = following
    return {(tuple(split[0] for split in chosen), differences)
            for chosen, differences in partial}


# ==================== Score-based parent search ====================

def score_based_parent_search(t, B, alphabet, weights, moduli,
                              inverse, bounds, smith=None, condition=None):
    A, D = t
    parents = set()
    chosen = []
    differences = []

    def lift(j):
        if j == len(A):
            parents.add((tuple(split[0] for split in chosen), tuple(differences)))
            return
        for split in buckets[j]:
            chosen.append(split)
            if j < 2:
                lift(j+1)
            else:
                v = parent_rhs(D[j-2], *chosen[-3:])
                for d in difference_vectors(v, inverse, bounds, smith, condition,
                                            known_solvable=True):
                    differences.append(d)
                    lift(j+1)
                    differences.pop()
            chosen.pop()

    for scores in admissible_score_sequences(t, B, alphabet, weights, moduli):
        buckets = [B[a][q] for a, q in zip(A, scores)]
        lift(0)
    return parents


# ==================== Sieve 2 ====================

def sieve2(t, alphabet):
    # True means keep. False means discard as non-genuine.
    A, D = t
    k = len(A)-1
    P = [parikh(a, alphabet) for a in A]
    nonempty = any(any(d) for d in D) or any(A[1:-1])
    states = {(x, y, bool(nonempty or (x == 0 and A[0] != '')))
              for x, y in product((0, 1), repeat=2)}
    for i in range(1, k):
        following = set()
        for x, y, nonempty in states:
            for z in (0, 1):
                difference = tuple(D[i-1][j]-(1-x)*P[i-1][j]
                                   +(1-2*y)*P[i][j]+z*P[i+1][j]
                                   for j in range(len(alphabet)))
                if not any(difference):
                    following.add((y, z, nonempty))
        states = following
        if not states:
            return True
    return not any(nonempty or (z == 1 and A[k] != '')
                   for y, z, nonempty in states)


# ==================== Ancestor search ====================

def ancestor_search(t, parent_search, alphabet, use_sieve2=False):
    # The starting template is included. For Sieve 2, start from T_k.
    ancestors = {t}
    waiting = deque([t])
    while waiting:
        child = waiting.popleft()
        for parent in parent_search(child):
            if use_sieve2 and not sieve2(parent, alphabet):
                continue
            if parent not in ancestors:
                ancestors.add(parent)
                waiting.append(parent)
    return ancestors


# ==================== Section 10.1: Canonical scoring ====================

def section_10_1():
    h = {'a': 'aaaab', 'b': 'abbab'}
    print_morphism(h, 5)
    alphabet = ('a', 'b')
    M = frequency_matrix(h, alphabet)
    inverse = M.inv()

    splits = (
        ('a', '',      '', 'aaaab'),
        ('a', 'a',     '', 'aaab'),
        ('b', 'ab',    '', 'bab'),
        ('a', 'aaa',   '', 'ab'),
        ('b', 'abba',  '', 'b'),
        ('a', 'aaaab', '', ''),
    )
    print('Child template: T5, with empty boundary letters and zero differences.')
    print()
    print('Split sequence (A_i, p_i, a_i, s_i):')
    for i, (A, p, a, s) in enumerate(splits):
        print(f'{i}: ({A}, {p or "ε"}, {a or "ε"}, {s or "ε"})')

    g = tuple(len(p) % 5 for A, p, a, s in splits)
    sh = tuple((p.count('a') + 6*p.count('b')) % 10
               for A, p, a, s in splits)
    print()
    print('Non-canonical prefix scores modulo 5:', g)
    print('Canonical prefix scores modulo 10:', sh)
    print()
    display(Math('M = ' + latex(M) + r',\qquad M^{-1} = ' + latex(inverse)))

    parikh_splits = tuple((A, parikh(p, alphabet), parikh(s, alphabet))
                          for A, p, a, s in splits)
    print()
    print('Parent equations, with coordinates ordered as (a, b):')
    display(Math(r'D_i M = v_i,\qquad D_i = v_i M^{-1}'))
    rows = []
    for i in range(4):
        v = Matrix([parent_rhs((0, 0), *parikh_splits[i:i+3])])
        D = v * inverse
        rows.append(f'v_{{{i+1}}} &= ' + latex(v)
                    + rf',\qquad & D_{{{i+1}}} &= ' + latex(D))
    display(Math(r'\begin{aligned}' + r'\\'.join(rows) + r'\end{aligned}'))
    print()
    print('This split sequence does not define an ordinary parent.')


# ==================== Section 10.2: The T8 example ====================

def section_10_2(full_search=False, ancestors=False):
    h = {'a': 'abaaaba', 'b': 'babab'}
    print_morphism(h, 8)
    alphabet = ('a', 'b')
    k = 8
    t = zero_template(k, alphabet)
    M = frequency_matrix(h, alphabet)
    inverse = prepare_inverse(M)
    weights = {'': (0,), 'a': (1,), 'b': (3,)}
    moduli = (11,)
    S = split_sets(h, alphabet)
    B = score_buckets(S, alphabet, weights, moduli)
    B1 = nontrivial_buckets(B)

    b = {q[0]: splits for q, splits in B[''].items()}
    b1 = {q[0]: splits for q, splits in B1[''].items()}
    progressions = []
    for q0 in b:
        for q1 in b:
            q = tuple((q0 + i*(q1-q0)) % 11 for i in range(k+1))
            if all(x in b for x in q):
                progressions.append(q)

    nonconstant = sorted(q for q in progressions if len(set(q)) > 1)
    full_count = sum(prod(len(b[x]) for x in q) for q in progressions)
    nt_count = sum(prod(len(b1[x]) for x in q) for q in progressions)
    print('Prefix scores:', sorted(b))
    print()
    print('Score, full bucket size, non-trivial bucket size:')
    for q in sorted(b):
        print(q, len(b[q]), len(b1[q]))
    print()
    print('Nonconstant progressions:', nonconstant)
    full_splits = sum(map(len, b.values()))
    nontrivial_splits = sum(map(len, b1.values()))
    print()
    print(f'For ε, the full split set contains {full_splits} splits; '
          f'after Sieve 1, {nontrivial_splits} remain.')
    print(f'All {k+1} boundary entries of T{k} are ε. '
          'Before applying score conditions, the Cartesian products therefore have sizes:')
    print(f'Full split family: {full_splits}^{k+1} = '
          f'{full_splits**(k+1):,} split tuples.')
    print(f'After Sieve 1: {nontrivial_splits}^{k+1} = '
          f'{nontrivial_splits**(k+1):,} split tuples.')
    print()
    print('Score-stage update upper bound:', (k-1)*len(b)**2)
    print('Compatible split tuples: full / Sieve 1:', full_count, nt_count)

    bounds = difference_bounds(h, inverse)

    def parents(child):
        return score_based_parent_search(
            child, B, alphabet, weights, moduli, inverse, bounds)

    def parents1(child):
        return score_based_parent_search(
            child, B1, alphabet, weights, moduli, inverse, bounds)

    P1 = parents1(t)
    P2 = {T for T in P1 if sieve2(T, alphabet)}
    print()
    print('Parents after Sieve 1 / both sieves:', len(P1), len(P2))
    if full_search:
        P = parents(t)
        print('Full parent set:', len(P))
    if ancestors:
        # No full ancestor calculation for T8.
        A1 = ancestor_search(t, parents1, alphabet)
        A2 = ancestor_search(t, parents1, alphabet, use_sieve2=True)
        print()
        print('Ancestors after Sieve 1 / both sieves:', len(A1), len(A2))


# ==================== Section 10.3: Keranen's morphism ====================

def section_10_3(ancestors=False):
    alphabet = tuple('abcd')
    word = (
        'abcacdcbcdcadcdbdabacabadbabcbdbcbacbcdcacb'
        'abdabacadcbcdcacdbcbacbcdcacdcbdcdadbdcbca'
    )
    h = {alphabet[i]: ''.join(alphabet[(alphabet.index(a)+i) % 4]
                              for a in word)
         for i in range(4)}
    print_morphism(h, 2)
    t = zero_template(2, alphabet)
    M = frequency_matrix(h, alphabet)
    inverse = prepare_inverse(M)
    weights = {'': (0,), 'a': (42904,), 'b': (4059,),
               'c': (41119,), 'd': (42734,)}
    moduli = (43435,)
    S = split_sets(h, alphabet)
    B = score_buckets(S, alphabet, weights, moduli)
    B1 = nontrivial_buckets(B)

    b = {q[0]: splits for q, splits in B[''].items()}
    b1 = {q[0]: splits for q, splits in B1[''].items()}
    progressions = [(x, y, (2*y-x) % 43435)
                    for x in b for y in b if (2*y-x) % 43435 in b]
    nonconstant = [q for q in progressions if len(set(q)) > 1]
    up_to_reversal = {min(q, q[::-1]) for q in nonconstant}
    full_count = sum(prod(len(b[x]) for x in q) for q in progressions)
    nt_count = sum(prod(len(b1[x]) for x in q) for q in progressions)

    display(Math('M = ' + latex(M)))
    print()
    print('Buckets / progressions / nonconstant progressions:',
          len(b), len(progressions), len(nonconstant))
    print('Compatible split triples: full / Sieve 1:', full_count, nt_count)
    print()
    print('Table 3 lists the prefix-score buckets for ε containing more than one split.')
    print('Each row gives the score, the number of splits in its bucket, '
          'and their prefix locations (A, j), where j is the prefix length in h(A).')
    print('(ε, 0) denotes the empty split; (A, 0) and (A, 85) are the endpoint splits for a source letter A.')
    print('The last column counts nonconstant three-term arithmetic progressions (APs) '
          'in the prefix-score set modulo 43435 that contain the row’s score.')
    print('An AP and its reversal are counted once. Constant APs are excluded.')
    print()
    print('Table 3: score, bucket size, prefix locations, nonconstant APs up to reversal')
    for q in sorted(b):
        if len(b[q]) > 1:
            locations = [(A or 'ε', sum(p)) for A, p, s in b[q]]
            ap_count = sum(q in progression for progression in up_to_reversal)
            print(q, len(b[q]), locations, ap_count)

    bounds = difference_bounds(h, inverse)

    def parents(child):
        return score_based_parent_search(
            child, B, alphabet, weights, moduli, inverse, bounds)

    def parents1(child):
        return score_based_parent_search(
            child, B1, alphabet, weights, moduli, inverse, bounds)

    P = parents(t)
    P1 = parents1(t)
    P2 = {T for T in P1 if sieve2(T, alphabet)}
    print()
    print('Parents: full / Sieve 1 / both:', len(P), len(P1), len(P2))
    if ancestors:
        A = ancestor_search(t, parents, alphabet)
        A1 = ancestor_search(t, parents1, alphabet)
        A2 = ancestor_search(t, parents1, alphabet, use_sieve2=True)
        print()
        print('Ancestors: full / Sieve 1 / both:', len(A), len(A1), len(A2))


# ==================== Section 10.4: Rao--Rosenfeld ====================

def rr_condition(D):
    a, b, c, d, e, f = D
    return (abs(-2*a+c-d+2*f) <= 6
            and abs(c+f-a-b) <= 2
            and abs(c+f-d-e) <= 4)


def rr_bounds(v):
    a, b, c, d, e, f = v
    if abs(2*b-c-d) > 2:
        return (0,)*6
    x = 2*a + 3*b - 2*c - 2*d
    E_min = int(ceiling(Rational(x-4, 3)))
    E_max = int(floor(Rational(x+4, 3)))
    F_min = int(ceiling(Rational(-6-b+2*c, 3)))
    F_max = int(floor(Rational(6-b+2*c, 3)))
    corners = []
    for E in (E_min, E_max):
        for F in (F_min, F_max):
            corners.append((-E-F+a+b-d, F-b+d,
                            -E-F+a+2*b-c-d, E-a-b+c+d, E, F))
    return tuple(max(abs(D[j]) for D in corners) for j in range(6))


def section_10_4(ancestors=False):
    h = {'a': 'ace', 'b': 'adf', 'c': 'bdf',
         'd': 'bdc', 'e': 'afe', 'f': 'bce'}
    print_morphism(h, 2)
    alphabet = tuple('abcdef')
    t = zero_template(2, alphabet)
    M = frequency_matrix(h, alphabet)
    smith = prepare_smith(M)
    inverse = None
    weights = {'': (0,0), 'a': (1,0), 'b': (1,0), 'c': (0,1),
               'd': (-1,-1), 'e': (-1,-1), 'f': (0,1)}
    moduli = (0, 0)
    S = split_sets(h, alphabet)
    B = score_buckets(S, alphabet, weights, moduli)
    B1 = nontrivial_buckets(B)

    b = B['']
    progressions = [(x, y, tuple(2*y[j]-x[j] for j in range(2)))
                    for x in b for y in b
                    if tuple(2*y[j]-x[j] for j in range(2)) in b]
    print('Prefix scores and bucket sizes:')
    for q in sorted(b):
        print(q, len(b[q]))
    print()
    print('Admissible score triples:', progressions)

    def parents(child):
        return score_based_parent_search(
            child, B, alphabet, weights, moduli, inverse, rr_bounds,
            smith=smith, condition=rr_condition)

    def parents1(child):
        return score_based_parent_search(
            child, B1, alphabet, weights, moduli, inverse, rr_bounds,
            smith=smith, condition=rr_condition)

    P = parents(t)
    P1 = parents1(t)
    P2 = {T for T in P1 if sieve2(T, alphabet)}
    print()
    print('Parents: full / Sieve 1 / both:', len(P), len(P1), len(P2))
    if ancestors:
        A = ancestor_search(t, parents, alphabet)
        A1 = ancestor_search(t, parents1, alphabet)
        A2 = ancestor_search(t, parents1, alphabet, use_sieve2=True)
        print()
        print('Ancestors: full / Sieve 1 / both:', len(A), len(A1), len(A2))


# ==================== Table 1: aaab / abb, k = 4 ====================

def table_T4():
    h = {'a': 'aaab', 'b': 'abb'}
    print_morphism(h, 4)
    alphabet = ('a', 'b')
    t = zero_template(4, alphabet)
    M = frequency_matrix(h, alphabet)
    inverse = prepare_inverse(M)
    weights = {'': (0,), 'a': (3,), 'b': (1,)}
    moduli = (5,)
    S = split_sets(h, alphabet)
    B = score_buckets(S, alphabet, weights, moduli)
    B1 = nontrivial_buckets(B)

    bounds = difference_bounds(h, inverse)

    def parents(child):
        return score_based_parent_search(
            child, B, alphabet, weights, moduli, inverse, bounds)

    def parents1(child):
        return score_based_parent_search(
            child, B1, alphabet, weights, moduli, inverse, bounds)

    P = parents(t)
    P1 = parents1(t)
    P2 = {T for T in P1 if sieve2(T, alphabet)}
    print()
    print('Parents: full / Sieve 1 / both:', len(P), len(P1), len(P2))
    A = ancestor_search(t, parents, alphabet)
    A1 = ancestor_search(t, parents1, alphabet)
    A2 = ancestor_search(t, parents1, alphabet, use_sieve2=True)
    print()
    print('Ancestors: full / Sieve 1 / both:', len(A), len(A1), len(A2))


# ==================== Table 1: T14, sieved ancestors only ====================

def table_T14():
    h = {'a': 'abaabaababa', 'b': 'babababab'}
    print_morphism(h, 14)
    alphabet = ('a', 'b')
    t = zero_template(14, alphabet)
    M = frequency_matrix(h, alphabet)
    inverse = prepare_inverse(M)
    weights = {'': (0,), 'a': (1,), 'b': (3,)}
    moduli = (19,)
    S1 = split_sets(h, alphabet, nontrivial=True)
    B1 = score_buckets(S1, alphabet, weights, moduli)
    bounds = difference_bounds(h, inverse)


    def parents1(child):
        return score_based_parent_search(
            child, B1, alphabet, weights, moduli, inverse, bounds)

    # Only sieved ancestor searches; no separate parent calculation.
    A1 = ancestor_search(t, parents1, alphabet)
    A2 = ancestor_search(t, parents1, alphabet, use_sieve2=True)
    print('T14 ancestors after Sieve 1 / both sieves:', len(A1), len(A2))


# ==================== Table 1: aaaab / ababab, k = 5 ====================

def table_T5():
    h = {'a': 'aaaab', 'b': 'ababab'}
    print_morphism(h, 5)
    alphabet = ('a', 'b')
    t = zero_template(5, alphabet)
    M = frequency_matrix(h, alphabet)
    inverse = prepare_inverse(M)
    weights = {'': (0,), 'a': (1,), 'b': (5,)}
    moduli = (9,)
    S = split_sets(h, alphabet)
    B = score_buckets(S, alphabet, weights, moduli)
    B1 = nontrivial_buckets(B)

    bounds = difference_bounds(h, inverse)

    def parents(child):
        return score_based_parent_search(
            child, B, alphabet, weights, moduli, inverse, bounds)

    def parents1(child):
        return score_based_parent_search(
            child, B1, alphabet, weights, moduli, inverse, bounds)

    P = parents(t)
    P1 = parents1(t)
    P2 = {T for T in P1 if sieve2(T, alphabet)}
    print()
    print('Parents: full / Sieve 1 / both:', len(P), len(P1), len(P2))
    A = ancestor_search(t, parents, alphabet)
    A1 = ancestor_search(t, parents1, alphabet)
    A2 = ancestor_search(t, parents1, alphabet, use_sieve2=True)
    print()
    print('Ancestors: full / Sieve 1 / both:', len(A), len(A1), len(A2))


# ==================== Table 1: Andrade--Mol, k = 5 ====================

def table_andrade_mol():
    h = {'a': 'aaaab', 'b': 'abbab'}
    print_morphism(h, 5)
    alphabet = ('a', 'b')
    t = zero_template(5, alphabet)
    M = frequency_matrix(h, alphabet)
    inverse = prepare_inverse(M)
    weights = {'': (0,), 'a': (1,), 'b': (6,)}
    moduli = (10,)
    S = split_sets(h, alphabet)
    B = score_buckets(S, alphabet, weights, moduli)
    B1 = nontrivial_buckets(B)

    bounds = difference_bounds(h, inverse)

    def parents(child):
        return score_based_parent_search(
            child, B, alphabet, weights, moduli, inverse, bounds)

    def parents1(child):
        return score_based_parent_search(
            child, B1, alphabet, weights, moduli, inverse, bounds)

    P = parents(t)
    P1 = parents1(t)
    P2 = {T for T in P1 if sieve2(T, alphabet)}
    print()
    print('Parents: full / Sieve 1 / both:', len(P), len(P1), len(P2))
    A = ancestor_search(t, parents, alphabet)
    A1 = ancestor_search(t, parents1, alphabet)
    A2 = ancestor_search(t, parents1, alphabet, use_sieve2=True)
    print()
    print('Ancestors: full / Sieve 1 / both:', len(A), len(A1), len(A2))


# ==================== Run all sections ====================

def print_eq(c):
    width = 70
    print()
    print('=' * width)
    print(f' {c} '.center(width, '='))
    print('=' * width)
    print()


if __name__ == '__main__':
    print_eq('10.1')
    section_10_1()

    print_eq('10.2')
    section_10_2(full_search=True, ancestors=True)

    print_eq('10.3')
    section_10_3(ancestors=True)

    print_eq('10.4')
    section_10_4(ancestors=True)

    print_eq('Table 1: T4')
    table_T4()

    print_eq('Table 1: T14')
    table_T14()

    print_eq('Table 1: T5')
    table_T5()

    print_eq('Table 1: Andrade-Mol')
    table_andrade_mol()

# T8: full parents, but only sieved ancestors.
# T14: only sieved ancestors; no separate parent calculation.



================================ 10.1 ================================



<IPython.core.display.Math object>


Child template: T5, with empty boundary letters and zero differences.

Split sequence (A_i, p_i, a_i, s_i):
0: (a, ε, ε, aaaab)
1: (a, a, ε, aaab)
2: (b, ab, ε, bab)
3: (a, aaa, ε, ab)
4: (b, abba, ε, b)
5: (a, aaaab, ε, ε)

Non-canonical prefix scores modulo 5: (0, 1, 2, 3, 4, 0)
Canonical prefix scores modulo 10: (0, 1, 7, 3, 4, 0)



<IPython.core.display.Math object>


Parent equations, with coordinates ordered as (a, b):


<IPython.core.display.Math object>

... (13 outputs left out)

<IPython.core.display.Math object>



Parents: full / Sieve 1 / both: 16163 247 113

Ancestors: full / Sieve 1 / both: 16163 319 113

======================== Table 1: Andrade-Mol ========================



<IPython.core.display.Math object>



Parents: full / Sieve 1 / both: 15625 129 3

Ancestors: full / Sieve 1 / both: 15625 129 3
